# DAI Mission — Proposal Template
**Data & AI in Economics | TU Dortmund**

This notebook is your team's mission proposal. Fill in every section before submission. Once approved, you will extend this same notebook into your final deliverable.

> **Team size:** 2–3 students  
> **Deliverable:** This Jupyter Notebook (proposal → final submission in one file)


## 1. Team

| Role | Name | Student ID |
|------|------|------------|
| Lead | Ananya Pal | |
| Member | Saizel Pathania | |
| Member *(optional)* | | |


## 2. Mission Title & Research Question

**Title:** *Geopolitical Conflict, Exchange Rates, and Financial Market Stability: Evidence from Global Exchange Rates and Major Equity Indices (2021–2026)*

**Research question:**  
*How did major geopolitical conflicts and periods of regional instability between 2021 and 2026 affect exchange rates and major stock-market indices, and what relationships can be observed between currency depreciation, market volatility, and investor behavior?*

**Why it matters:**  
*Geopolitical conflicts create uncertainty that affects financial markets through changes in investor expectations, capital flows, and risk perceptions. Exchange rates often react immediately to such shocks, while equity markets may experience changes in returns and volatility. Understanding these relationships provides insight into how financial markets price geopolitical risk and how different regions respond to periods of instability.*


## 3. Data

**Source(s):**  
*Name each dataset, its provider, URL or access method, and licence/terms of use.*
1. WRDS Global Exchange Rates (wrds_g_exrate) : https://wrds-www.wharton.upenn.edu/pages/get-data/compustat-capital-iq-standard-poors/compustat/global-daily/exchange-rates/
- Daily exchange rates.
- Focus on IRR, EUR, GBP and other relevant currencies.
- Used to measure currency depreciation and exchange-rate volatility.
2. WRDS Global Index Daily Prices (g_idx_daily)
- Daily market-index prices.
- Used to compute returns and volatility measures.
3. WRDS Global Index Metadata (g_idx_index) 
- Provides index names, tickers and geographic information.
- Used to identify relevant indices such as S&P, DAX and other regional benchmarks.


In [45]:
import wrds 

db = wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [46]:
# ============================================================
# DATASET 1: Global Exchange Rates
# WRDS Table:
#   comp_global_daily.wrds_g_exrate
#
# Purpose:
# - Core dataset for measuring currency movements.
# - Supports analysis of IRR depreciation during periods of
#   geopolitical instability.
# - Also provides benchmark currencies such as EUR and GBP.
#
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Available currencies
# ------------------------------------------------------------

currencies = db.raw_sql("""
SELECT DISTINCT curd
FROM comp_global_daily.wrds_g_exrate
ORDER BY curd
""")

print(f"Number of currencies: {len(currencies)}")
display(currencies.head(50))

# ------------------------------------------------------------
# Pull exchange rates for project period
# ------------------------------------------------------------

fx = db.raw_sql("""
SELECT
    curd,
    datadate,
    exratd_tousd,
    exratd_fromusd
FROM comp_global_daily.wrds_g_exrate
WHERE datadate >= '2021-01-01'
ORDER BY curd, datadate
""")

fx["datadate"] = pd.to_datetime(fx["datadate"])

# ------------------------------------------------------------
# Basic inspection
# ------------------------------------------------------------

print("Shape:")
print(fx.shape)

print("\nColumns:")
print(fx.columns.tolist())

print("\nData types:")
display(fx.dtypes)

print("\nFirst rows:")
display(fx.head())

print("\nMissing values:")
display(fx.isnull().sum())

print("\nNumerical summary:")
display(fx.describe())

# ------------------------------------------------------------
# Relevant currencies
# ------------------------------------------------------------

availability = db.raw_sql("""
SELECT
    curd,
    MIN(datadate) AS first_date,
    MAX(datadate) AS last_date,
    COUNT(*)      AS n_obs
FROM comp_global_daily.wrds_g_exrate
WHERE curd IN ('IRR','EUR','GBP')
GROUP BY curd
ORDER BY curd
""")

print("\nCoverage of project currencies:")
display(availability)

Number of currencies: 205


,curd
0,AED
1,AFA
2,AFN
3,ALL
4,AMD
5,ANG
6,AOA
7,AON
8,AOR
9,ARA


Shape:
(302290, 4)

Columns:
['curd', 'datadate', 'exratd_tousd', 'exratd_fromusd']

Data types:


curd              string[python]
datadate          datetime64[ns]
exratd_tousd             Float64
exratd_fromusd           Float64
dtype: object


First rows:


...

curd              0
datadate          0
exratd_tousd      0
exratd_fromusd    0
dtype: int64


Numerical summary:


,datadate,exratd_tousd,exratd_fromusd
count,302290,302290.0,302290.0
mean,2023-09-12 02:45:53.620695552,0.492369,4869.661678
min,2021-01-01 00:00:00,0.0,0.021507
25%,2022-05-06 00:00:00,0.002582,4.006116
50%,2023-09-11 00:00:00,0.02754,36.310483
75%,2025-01-18 00:00:00,0.249618,387.224298
max,2026-05-29 00:00:00,46.496599,4172370.062139
std,NaN,3.290471,90450.78697



Coverage of project currencies:


,curd,first_date,last_date,n_obs
0,EUR,1985-12-31,2026-05-29,13647
1,GBP,1982-02-01,2026-05-29,15538
2,IRR,1982-02-01,2026-05-29,15358


In [47]:
# ============================================================
# DATASET 2A: Index Metadata
# WRDS Table:
#   comp_global_daily.g_idx_index
#
# Purpose:
# - Identifies available stock-market indices.
# - Used to locate S&P 500, DAX and other regional indices.
# - Provides metadata needed to interpret g_idx_daily.
#
# ============================================================

idx_meta = db.raw_sql("""
SELECT *
FROM comp_global_daily.g_idx_index
""")

print("Shape:")
print(idx_meta.shape)

print("\nColumns:")
print(idx_meta.columns.tolist())

print("\nMissing values:")
display(idx_meta.isnull().sum())

print("\nSample:")
display(idx_meta.head())

# ------------------------------------------------------------
# Candidate indices
# ------------------------------------------------------------

print("\nPotential S&P indices:")
display(
    idx_meta[
        idx_meta["conm"].str.contains(
            "S&P",
            case=False,
            na=False
        )
    ][["gvkeyx", "conm", "tic", "indexgeo"]]
)

print("\nPotential DAX indices:")
display(
    idx_meta[
        idx_meta["conm"].str.contains(
            "DAX",
            case=False,
            na=False
        )
    ][["gvkeyx", "conm", "tic", "indexgeo"]]
)

print("\nIran-related indices:")
display(
    idx_meta[
        idx_meta["conm"].str.contains(
            "Iran|Tehran",
            case=False,
            na=False
        )
    ][["gvkeyx", "conm", "tic", "indexgeo"]]
)

Shape:
(735, 14)

Columns:
['conm', 'gvkeyx', 'idx13key', 'idxcstflg', 'idxstat', 'indexcat', 'indexgeo', 'indexid', 'indextype', 'indexval', 'spii', 'spmi', 'tic', 'tici']

Missing values:


conm           0
gvkeyx         0
idx13key       0
idxcstflg      0
idxstat        1
indexcat       0
indexgeo      80
indexid        0
indextype      0
indexval       0
spii         735
spmi         735
tic            0
tici           1
dtype: int64


Sample:


,conm,gvkeyx,idx13key,idxcstflg,idxstat,indexcat,indexgeo,indexid,indextype,indexval,spii,spmi,tic,tici
0,Malta Stock Exchange Index,115118,I192956,N,A,EXCHG,MLT,MLT,COMPOSITE,MLT,<NA>,<NA>,I3MLT002,I3MLT002
1,Affarsvarlden General Index,150001,I000234,N,A,EXCHG,SWE,SWE,COMPOSITE,SWE,<NA>,<NA>,I3SWE001,I3SWE001
2,Banco Totta & Acores Index,150002,I000220,N,I,EXCHG,PRT,PRT,COMPOSITE,PRT,<NA>,<NA>,I3PRT001,I3PRT001
3,BCI All-Share Index,150003,I000090,N,A,EXCHG,ITA,BCI,COMPOSITE,BCI,<NA>,<NA>,I3ITA002,I3ITA002
4,FTSE All-World Index,150004,I000294,N,A,FT,<NA>,ALL,REGIONAL,ALL,<NA>,<NA>,I6UNK001,I6UNK001



Potential S&P indices:


,gvkeyx,conm,tic,indexgeo
72,150099,S&P 700 Index,I6UNK107,<NA>
75,150147,S&P ASX 20 Index,I2AUS002,AUS
83,150155,S&P ASX ALL AUSTRALIAN 50 INDX,I2AUS010,AUS
155,150912,S&P Euro Plus Index,I6UNK084,EUR
156,150913,S&P Euro Index,I6UNK085,EUR
157,150915,S&P Latin America 40 Index,I6UNK113,LAC
158,150916,S&P/Topix Index (Japan) Index,I2JPN011,JPN
159,150917,S&P United Kingdom Index,I3GBR065,GBR
160,150918,S&P Global 1200 Index,I6UNK112,<NA>
161,150919,S&P Global 100 Index,I6UNK111,<NA>



Potential DAX indices:


,gvkeyx,conm,tic,indexgeo
56,150007,Composite DAX Index,I3DEU003,DEU
69,150095,Deutscher Aktienindex (DAX) Index,I3DEU001,DEU
175,150356,HDAX,I3DEU005,DEU
297,153256,Deutscher Aktien MDAX (Perf) Index,I3DEU009,DEU
638,153432,Deutschier Aktien TECDAX (Perf),I3DEU029,DEU
689,160133,Deutscher Aktien MDAX (Price) Index,I3DEU004,DEU



Iran-related indices:


,gvkeyx,conm,tic,indexgeo


In [48]:
# ============================================================
# DATASET 2B: Daily Index Prices
# WRDS Table:
#   comp_global_daily.g_idx_daily
#
# Purpose:
# - Measures stock-market performance around conflict events.
# - Can be linked to g_idx_index through gvkeyx.
# ============================================================

idx_daily = db.raw_sql("""
SELECT *
FROM comp_global_daily.g_idx_daily
LIMIT 10000
""")

print("Shape:")
print(idx_daily.shape)

print("\nColumns:")
print(idx_daily.columns.tolist())

print("\nData types:")
display(idx_daily.dtypes)

print("\nMissing values:")
display(idx_daily.isnull().sum())

print("\nSample:")
display(idx_daily.head())

print("\nNumerical summary:")
display(idx_daily.describe())

Shape:
(10000, 10)

Columns:
['gvkeyx', 'dvpsxd', 'newnum', 'oldnum', 'prccd', 'prccddiv', 'prccddivn', 'prchd', 'prcld', 'datadate']

Data types:


gvkeyx       string[python]
dvpsxd       string[python]
newnum       string[python]
oldnum       string[python]
prccd               Float64
prccddiv     string[python]
prccddivn    string[python]
prchd        string[python]
prcld        string[python]
datadate     string[python]
dtype: object


Missing values:


gvkeyx           0
dvpsxd       10000
newnum       10000
oldnum       10000
prccd            0
prccddiv     10000
prccddivn    10000
prchd        10000
prcld        10000
datadate         0
dtype: int64


Sample:


,gvkeyx,dvpsxd,newnum,oldnum,prccd,prccddiv,prccddivn,prchd,prcld,datadate
0,115114,<NA>,<NA>,<NA>,96.41,<NA>,<NA>,<NA>,<NA>,1991-10-31
1,115114,<NA>,<NA>,<NA>,79.21,<NA>,<NA>,<NA>,<NA>,1991-11-30
2,115114,<NA>,<NA>,<NA>,72.52,<NA>,<NA>,<NA>,<NA>,1991-12-31
3,115114,<NA>,<NA>,<NA>,71.61,<NA>,<NA>,<NA>,<NA>,1992-01-31
4,115114,<NA>,<NA>,<NA>,71.71,<NA>,<NA>,<NA>,<NA>,1992-02-28



Numerical summary:


,prccd
count,10000.0
mean,754.255464
std,1272.471913
min,23.0
25%,53.5175
50%,104.2
75%,1009.41
max,6641.873


## 4. Planned Methods

Your mission **must** apply at least one technique from **each** of the three blocks below. Tick the ones you plan to use and briefly justify the choice.

### 4a. Causal Inference
- [ - ] **Causal graph / DAG (DoWhy)**
- [ - ] **Backdoor adjustment**
- [ ] Instrumental variable
- [ ] Propensity score stratification
- [ ] Other: ___

*Justification:*
DAG for reasoning of impact on oil prices, inflation, exchange rates and stock market indices.
Backdoor adjustment to estimate the relationship between conflict periods and market variables.

### 4b. Supervised Learning
- [ - ] **Linear / Ridge / Lasso regression**
- [ ] Logistic regression
- [ ] k-Nearest Neighbors
- [ ] Support Vector Machine
- [ - ] **Decision Tree / Random Forest**
- [ ] Neural network (regression or classification)
- [ ] Other: ___

*Justification:*
Linear and regularized regression models will be used to estimate and interpret relationships between macroeconomic indicators and exchange-rate or stock-index movements. Random Forest models will be included as nonlinear benchmarks to compare predictive performance and capture potential interaction effects between geopolitical and financial variables.

### 4c. Unsupervised Learning / Generative Models
- [ - ] **K-Means clustering**
- [ - ] **Hierarchical clustering**
- [ ] Variational autoencoder
- [ ] GAN
- [ ] Other: ___

*Justification:*
Clustering methods will be used to identify similarities in financial-market behavior across countries and time periods. K-Means and hierarchical clustering can help detect groups of stable versus high-volatility periods and reveal whether countries exposed to geopolitical instability exhibit similar exchange-rate and stock-market dynamics.

## 5. Evaluation Strategy

*How will you know if your mission succeeded? Describe:*

- The metric(s) you will use for each model (e.g. RMSE, accuracy, AUC, silhouette score).
- How you will validate causal claims (e.g. refutation tests, sensitivity analysis).
- Any baselines or benchmarks you will compare against.


## 6. Work Plan

| Step | Owner | Description |
|------|-------|-------------|
| 1 | | Data collection & cleaning |
| 2 | | EDA |
| 3 | | Causal inference block |
| 4 | | Supervised learning block |
| 5 | | Unsupervised / generative block |
| 6 | | Synthesis & write-up |


---
## 7. Results *(complete for final submission)*


### 7a. Causal Inference

In [49]:
# Causal inference analysis

### 7b. Supervised Learning

In [50]:
# Supervised learning analysis

### 7c. Unsupervised / Generative

In [51]:
# Unsupervised / generative analysis

## 8. Discussion & Conclusion *(complete for final submission)*

*Synthesise findings across all three method blocks. What does each lens reveal that the others miss? What are the limitations of your analysis?*
